# 08 - Quantitative Comparison with Pastor/SpeechXAI

This notebook loads the two result ZIP files directly with pandas, analyzes IEMOCAP and RAVDESS separately, and writes the report tables used in the progress report.

## How to download the result ZIPs

The result ZIPs are stored with Git LFS. Download them with Git LFS, otherwise Git may leave small pointer files instead of the real ZIP files.

For a fresh clone:

```bash
git lfs install
git clone --branch relevance_hubert_pipeline https://github.com/MateusWiteck/gradient_based_speach_xai.git
cd gradient_based_speach_xai
git lfs pull --include="*_speech_xai_results_*.zip"
```

For an existing clone:

```bash
git lfs install
git pull
git lfs pull --include="*_speech_xai_results_*.zip"
```

Check that the real ZIP files were downloaded:

```bash
git lfs ls-files
ls -lh *_speech_xai_results_*.zip
```

## Required result files

Place the result ZIP files in the project root, next to `README.md`:

```text
gradient_based_speach_xai/
|-- IEMOCAP_speech_xai_results_20260711_074459.zip
|-- RAVDESS_speech_xai_results_20260711_100706.zip
|-- README.md
|-- notebooks/
`-- src/
```

Do not extract the ZIP files for the default notebook path. The notebook reads `duration_matched_records.csv` directly from inside each ZIP using `pd.read_csv(...)`.

Generated comparison tables are written to:

```text
results/pastor_quantitative_comparison/
```

IEMOCAP is a partial run, so it is never pooled with RAVDESS.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

OUTPUT_DIR = ROOT / "results" / "pastor_quantitative_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for old_csv in OUTPUT_DIR.glob("*.csv"):
    old_csv.unlink()

VALID_K = [1, 2, 3, 5]
USECOLS = [
    "dataset", "audio_id", "k", "method", "method_family", "selection",
    "masked_duration", "confidence_drop", "prediction_flipped", "random_trial",
]

sources = pd.DataFrame([
    {
        "dataset": "IEMOCAP",
        "zip_file": "IEMOCAP_speech_xai_results_20260711_074459.zip",
        "csv_inside_zip": "iemocap_ravdess_duration_matched_speechxai_20260710_194505_901788/duration_matched_records.csv",
        "expected_audios": 5531,
        "scope": "partial",
        "note": "Partial IEMOCAP run; do not describe as full-dataset IEMOCAP.",
    },
    {
        "dataset": "RAVDESS",
        "zip_file": "RAVDESS_speech_xai_results_20260711_100706.zip",
        "csv_inside_zip": "ravdess_duration_matched_speechxai_20260711_092556_063830/duration_matched_records.csv",
        "expected_audios": 672,
        "scope": "available RAVDESS run",
        "note": "RAVDESS analysis is separate from IEMOCAP.",
    },
])

method_labels = {
    "speechxai_pastor_loo_words": "Pastor/SpeechXAI",
    "random_duration_matched_bins": "Random duration-matched",
    "level3": "Level3 relevance",
    "legrad_final_score_relu_attention_gradient_mean_layers_source_tokens": "LeGrad final score, mean layers",
    "legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens": "LeGrad final score, HuBERT-weighted layers",
    "legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens": "LeGrad layer-local, mean layers",
    "legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens": "LeGrad layer-local, HuBERT-weighted layers",
}

frames = []
quality_rows = []
for source in sources.itertuples(index=False):
    raw = pd.read_csv(
        f"zip://{source.csv_inside_zip}::{ROOT / source.zip_file}",
        usecols=USECOLS,
        low_memory=False,
    )
    raw["k"] = pd.to_numeric(raw["k"], errors="coerce")
    raw["masked_duration"] = pd.to_numeric(raw["masked_duration"], errors="coerce")
    raw["confidence_drop"] = pd.to_numeric(raw["confidence_drop"], errors="coerce")
    raw["random_trial"] = pd.to_numeric(raw["random_trial"], errors="coerce")
    raw["prediction_flipped"] = raw["prediction_flipped"].astype(str).str.lower().map(
        {"true": True, "false": False, "1": True, "0": False}
    )

    records = raw[(raw["dataset"] == source.dataset) & raw["k"].isin(VALID_K)].copy()
    records["k"] = records["k"].astype(int)
    frames.append(records)

    rows_per_audio = records.groupby("audio_id").size()
    quality_rows.append({
        "dataset": source.dataset,
        "scope": source.scope,
        "source_zip": source.zip_file,
        "expected_audios": source.expected_audios,
        "kept_audios": records["audio_id"].nunique(),
        "missing_audio_count": source.expected_audios - records["audio_id"].nunique(),
        "raw_rows": len(raw),
        "kept_rows": len(records),
        "excluded_rows": len(raw) - len(records),
        "full_156_row_audios": int((rows_per_audio == 156).sum()),
        "incomplete_audios": int((rows_per_audio != 156).sum()),
        "note": source.note,
    })

all_records = pd.concat(frames, ignore_index=True)
data_quality = pd.DataFrame(quality_rows)

random_trial_counts = (
    all_records[all_records["method"].eq("random_duration_matched_bins")]
    .groupby(["dataset", "audio_id", "k"])["random_trial"]
    .nunique(dropna=True)
)
if not random_trial_counts.empty:
    random_trials_per_audio = int(random_trial_counts.median())
    method_labels["random_duration_matched_bins"] = (
        f"Random duration-matched ({random_trials_per_audio} trials/audio)"
    )

display(data_quality)
display(Markdown("**Legend.** `kept_audios` is the number of usable audios in the ZIP after filtering to the intended dataset and k values; `full_156_row_audios` counts audios with all expected deletion rows."))

,dataset,scope,source_zip,expected_audios,kept_audios,missing_audio_count,raw_rows,kept_rows,excluded_rows,full_156_row_audios,incomplete_audios,note
0,IEMOCAP,partial,IEMOCAP_speech_xai_results_20260711_074459.zip,5531,5460,71,956431,851752,104679,5459,1,Partial IEMOCAP run; do not describe as full-d...
1,RAVDESS,available RAVDESS run,RAVDESS_speech_xai_results_20260711_100706.zip,672,671,1,104676,104676,0,671,0,RAVDESS analysis is separate from IEMOCAP.


**Legend.** `kept_audios` is the number of usable audios in the ZIP after filtering to the intended dataset and k values; `full_156_row_audios` counts audios with all expected deletion rows.

In [2]:
summary = (
    all_records.groupby(["dataset", "k", "method", "method_family", "selection"], as_index=False)
    .agg(
        audios=("audio_id", "nunique"),
        rows=("audio_id", "size"),
        mean_confidence_drop=("confidence_drop", "mean"),
        prediction_flip_rate=("prediction_flipped", "mean"),
        mean_masked_duration=("masked_duration", "mean"),
    )
)

report_keys = ["dataset", "k", "method", "method_family", "selection"]
report_methods = summary[
    summary["method"].isin(method_labels)
    & summary["selection"].isin(["top", "top_words", "random"])
].copy()

audio_method = (
    all_records.groupby(["dataset", "audio_id", "k", "method", "method_family", "selection"], as_index=False)
    .agg(confidence_drop=("confidence_drop", "mean"))
)
pastor_by_audio = (
    audio_method[audio_method["method_family"] == "speechxai_pastor"]
    [["dataset", "audio_id", "k", "confidence_drop"]]
    .rename(columns={"confidence_drop": "pastor_confidence_drop"})
)
paired = audio_method.merge(pastor_by_audio, on=["dataset", "audio_id", "k"], how="inner")
paired = paired[paired["method_family"] != "speechxai_pastor"].copy()
paired["delta_vs_pastor"] = paired["confidence_drop"] - paired["pastor_confidence_drop"]
paired["beats_pastor"] = paired["delta_vs_pastor"] > 0

paired_summary = (
    paired.groupby(report_keys, as_index=False)
    .agg(
        mean_confidence_drop=("confidence_drop", "mean"),
        pastor_mean_confidence_drop=("pastor_confidence_drop", "mean"),
        mean_delta_vs_pastor=("delta_vs_pastor", "mean"),
        beats_pastor_rate=("beats_pastor", "mean"),
    )
)
paired_summary["mean_delta_vs_pastor_percentage"] = np.where(
    paired_summary["pastor_mean_confidence_drop"].abs() > np.finfo(float).eps,
    100.0 * paired_summary["mean_delta_vs_pastor"] / paired_summary["pastor_mean_confidence_drop"],
    np.nan,
)
paired_summary["beats_pastor"] = paired_summary["mean_delta_vs_pastor"] > 0

pastor_summary = report_methods[report_methods["method_family"].eq("speechxai_pastor")].copy()
pastor_summary["pastor_mean_confidence_drop"] = pastor_summary["mean_confidence_drop"]
pastor_summary["mean_delta_vs_pastor"] = 0.0
pastor_summary["mean_delta_vs_pastor_percentage"] = 0.0
pastor_summary["beats_pastor_rate"] = np.nan
pastor_summary["beats_pastor"] = pd.NA

comparison_columns = [
    *report_keys,
    "pastor_mean_confidence_drop",
    "mean_delta_vs_pastor",
    "mean_delta_vs_pastor_percentage",
    "beats_pastor_rate",
    "beats_pastor",
]
comparison_summary = pd.concat(
    [pastor_summary[comparison_columns], paired_summary[comparison_columns]],
    ignore_index=True,
)

report_table = report_methods.merge(comparison_summary, on=report_keys, how="left")
report_table["method_label"] = report_table["method"].map(method_labels)
report_table = report_table[[
    "dataset", "k", "method_label",
    "mean_confidence_drop", "pastor_mean_confidence_drop",
    "mean_delta_vs_pastor", "mean_delta_vs_pastor_percentage",
    "beats_pastor", "prediction_flip_rate", "mean_masked_duration",
]].sort_values(["dataset", "k", "mean_confidence_drop"], ascending=[True, True, False])

best_relevance = (
    paired_summary[
        (paired_summary["method_family"] == "hubert_temporal_relevance")
        & (paired_summary["selection"] == "top")
    ]
    .sort_values(["dataset", "k", "mean_confidence_drop"], ascending=[True, True, False])
    .groupby(["dataset", "k"], as_index=False)
    .head(1)
)
best_relevance["method_label"] = best_relevance["method"].map(method_labels)

data_quality.to_csv(OUTPUT_DIR / "data_quality.csv", index=False)
report_table.to_csv(OUTPUT_DIR / "report_table_by_dataset_k.csv", index=False)


## IEMOCAP

Partial run only. The first table shows the best relevance method for each k; the second table contains all report methods.


In [3]:
display(best_relevance[best_relevance["dataset"] == "IEMOCAP"][[
    "k", "method_label", "mean_confidence_drop", "pastor_mean_confidence_drop",
    "mean_delta_vs_pastor", "mean_delta_vs_pastor_percentage", "beats_pastor_rate",
]].style.format({
    "mean_confidence_drop": "{:.4f}",
    "pastor_mean_confidence_drop": "{:.4f}",
    "mean_delta_vs_pastor": "{:+.4f}",
    "mean_delta_vs_pastor_percentage": "{:+.1f}%",
    "beats_pastor_rate": "{:.3f}",
}))

display(report_table[report_table["dataset"] == "IEMOCAP"].style.format({
    "mean_confidence_drop": "{:.4f}",
    "pastor_mean_confidence_drop": "{:.4f}",
    "mean_delta_vs_pastor": "{:+.4f}",
    "mean_delta_vs_pastor_percentage": "{:+.1f}%",
    "prediction_flip_rate": "{:.3f}",
    "mean_masked_duration": "{:.3f}",
}))


,k,method_label,mean_confidence_drop,pastor_mean_confidence_drop,mean_delta_vs_pastor,mean_delta_vs_pastor_percentage,beats_pastor_rate
13,1,Level3 relevance,0.1284,0.1146,+0.0138,+12.0%,0.553
32,2,Level3 relevance,0.1805,0.1586,+0.0219,+13.8%,0.560
51,3,Level3 relevance,0.2146,0.1832,+0.0314,+17.2%,0.564
70,5,Level3 relevance,0.2560,0.2069,+0.0491,+23.7%,0.571


,dataset,k,method_label,mean_confidence_drop,pastor_mean_confidence_drop,mean_delta_vs_pastor,mean_delta_vs_pastor_percentage,beats_pastor,prediction_flip_rate,mean_masked_duration
4,IEMOCAP,1,Level3 relevance,0.1284,0.1146,+0.0138,+12.0%,True,0.284,0.431
6,IEMOCAP,1,Pastor/SpeechXAI,0.1146,0.1146,+0.0000,+0.0%,,0.281,0.431
3,IEMOCAP,1,"LeGrad layer-local, HuBERT-weighted layers",0.1062,0.1146,-0.0084,-7.3%,False,0.245,0.431
2,IEMOCAP,1,"LeGrad layer-local, mean layers",0.1046,0.1146,-0.0100,-8.7%,False,0.241,0.431
1,IEMOCAP,1,"LeGrad final score, HuBERT-weighted layers",0.0847,0.1146,-0.0299,-26.1%,False,0.194,0.431
0,IEMOCAP,1,"LeGrad final score, mean layers",0.0825,0.1146,-0.0321,-28.0%,False,0.190,0.431
5,IEMOCAP,1,Random duration-matched (20 trials/audio),0.0462,0.1146,-0.0685,-59.7%,False,0.127,0.431
11,IEMOCAP,2,Level3 relevance,0.1805,0.1586,+0.0219,+13.8%,True,0.383,0.725
13,IEMOCAP,2,Pastor/SpeechXAI,0.1586,0.1586,+0.0000,+0.0%,,0.373,0.725
10,IEMOCAP,2,"LeGrad layer-local, HuBERT-weighted layers",0.1547,0.1586,-0.0039,-2.4%,False,0.338,0.725


## RAVDESS

Same calculations as IEMOCAP, kept separate.

In [4]:
display(best_relevance[best_relevance["dataset"] == "RAVDESS"][[
    "k", "method_label", "mean_confidence_drop", "pastor_mean_confidence_drop",
    "mean_delta_vs_pastor", "mean_delta_vs_pastor_percentage", "beats_pastor_rate",
]].style.format({
    "mean_confidence_drop": "{:.4f}",
    "pastor_mean_confidence_drop": "{:.4f}",
    "mean_delta_vs_pastor": "{:+.4f}",
    "mean_delta_vs_pastor_percentage": "{:+.1f}%",
    "beats_pastor_rate": "{:.3f}",
}))

display(report_table[report_table["dataset"] == "RAVDESS"].style.format({
    "mean_confidence_drop": "{:.4f}",
    "pastor_mean_confidence_drop": "{:.4f}",
    "mean_delta_vs_pastor": "{:+.4f}",
    "mean_delta_vs_pastor_percentage": "{:+.1f}%",
    "prediction_flip_rate": "{:.3f}",
    "mean_masked_duration": "{:.3f}",
}))


,k,method_label,mean_confidence_drop,pastor_mean_confidence_drop,mean_delta_vs_pastor,mean_delta_vs_pastor_percentage,beats_pastor_rate
85,1,"LeGrad layer-local, mean layers",0.1626,0.1088,+0.0538,+49.5%,0.723
104,2,"LeGrad layer-local, mean layers",0.3277,0.1986,+0.1291,+65.0%,0.808
125,3,"LeGrad layer-local, HuBERT-weighted layers",0.4128,0.2490,+0.1638,+65.8%,0.842
144,5,"LeGrad layer-local, HuBERT-weighted layers",0.4327,0.2272,+0.2055,+90.5%,0.842


,dataset,k,method_label,mean_confidence_drop,pastor_mean_confidence_drop,mean_delta_vs_pastor,mean_delta_vs_pastor_percentage,beats_pastor,prediction_flip_rate,mean_masked_duration
30,RAVDESS,1,"LeGrad layer-local, mean layers",0.1626,0.1088,+0.0538,+49.5%,True,0.265,0.388
31,RAVDESS,1,"LeGrad layer-local, HuBERT-weighted layers",0.1607,0.1088,+0.0519,+47.7%,True,0.253,0.388
29,RAVDESS,1,"LeGrad final score, HuBERT-weighted layers",0.1581,0.1088,+0.0494,+45.4%,True,0.240,0.388
28,RAVDESS,1,"LeGrad final score, mean layers",0.1576,0.1088,+0.0489,+44.9%,True,0.234,0.388
34,RAVDESS,1,Pastor/SpeechXAI,0.1088,0.1088,+0.0000,+0.0%,,0.164,0.388
32,RAVDESS,1,Level3 relevance,0.1040,0.1088,-0.0048,-4.4%,False,0.167,0.388
33,RAVDESS,1,Random duration-matched (20 trials/audio),0.0265,0.1088,-0.0822,-75.6%,False,0.061,0.388
37,RAVDESS,2,"LeGrad layer-local, mean layers",0.3277,0.1986,+0.1291,+65.0%,True,0.541,0.767
38,RAVDESS,2,"LeGrad layer-local, HuBERT-weighted layers",0.3267,0.1986,+0.1281,+64.5%,True,0.534,0.767
36,RAVDESS,2,"LeGrad final score, HuBERT-weighted layers",0.3212,0.1986,+0.1226,+61.7%,True,0.541,0.767


## Output Files

The notebook writes only the data-quality table and the report table. No separate decision table is generated.


In [5]:
display(Markdown(f"**Files written to `{OUTPUT_DIR}`:** `data_quality.csv` and `report_table_by_dataset_k.csv`."))


**Files written to `C:\Users\mateu\repos\gradient_based_speach_xai\results\pastor_quantitative_comparison`:** `data_quality.csv` and `report_table_by_dataset_k.csv`.